# Tutorial 7: Sequential Quadratic Programming (SQP)

**Prerequisites:** Tutorial 5 — Newton and Quasi-Newton Methods, Tutorial 6 — KKT Conditions  
**Up Next:** Tutorial 8 — Penalty and Augmented Lagrangian Methods

---

## Concept

In Tutorial 5 we saw that BFGS converges faster than gradient descent by building curvature information. In Tutorial 6 we introduced constraints and the KKT conditions. **Sequential Quadratic Programming (SQP)** brings these ideas together: at each iteration it solves a **quadratic subproblem** — a local quadratic model of the objective subject to linearized constraints — to find both a search direction *and* updated Lagrange multiplier estimates.

SQP is essentially Newton's method applied to the KKT system. It converges superlinearly near a solution and is the algorithm behind scipy's `SLSQP` solver.

## Key Takeaway

> **SQP solves constrained optimization by iteratively solving quadratic subproblems that approximate the original problem. It handles equality and inequality constraints natively and converges faster than penalty-based approaches.**

## Math Core

At iteration $k$, SQP solves the **quadratic subproblem**:

$$\min_{\mathbf{d}} \; \nabla f_k^T \mathbf{d} + \tfrac{1}{2} \mathbf{d}^T B_k \mathbf{d} \quad \text{s.t.} \quad g_i(\mathbf{x}_k) + \nabla g_i(\mathbf{x}_k)^T \mathbf{d} \le 0$$

where $B_k$ is a (quasi-Newton) approximation to the Hessian of the Lagrangian, and $\mathbf{d}$ is the step direction. The solution gives both $\mathbf{d}_k$ and the Lagrange multipliers $\lambda_k$ for the linearized constraints. Then:

$$\mathbf{x}_{k+1} = \mathbf{x}_k + \alpha_k \mathbf{d}_k$$

with $\alpha_k$ chosen by a line search on a merit function.

## Code

We compare the unconstrained BFGS solution from Tutorial 5 against SLSQP with the Grashof constraint from Tutorial 6.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from dms.curves import CompareCurves
from scipy.optimize import minimize
%matplotlib inline

# Constants (same as Tutorial 1)
L0, L1 = 1.0, 1.0
PX, PY = 0.5, 0.3
TARGETS = np.array([
    [1.0, 1.5], [0.75, 1.5], [0.5, 1.5],
    [0.25, 1.5], [0.0, 1.5], [-0.25, 1.5]
])
THETAS = np.linspace(np.deg2rad(60), np.deg2rad(120), len(TARGETS))

In [ ]:
def fourbar_fk(theta1, l0, l1, l2, l3):
    """Cosine-law closed-form FK for four-bar linkage."""
    ax, ay = l1 * np.cos(theta1), l1 * np.sin(theta1)
    dx, dy = ax - l0, ay
    d = np.sqrt(dx**2 + dy**2)
    cos_alpha = (l3**2 + d**2 - l2**2) / (2 * l3 * d)
    if np.abs(cos_alpha) > 1:
        return None
    alpha = np.arccos(np.clip(cos_alpha, -1, 1))
    beta = np.arctan2(dy, dx)
    theta3 = beta + alpha
    bx = l0 + l3 * np.cos(theta3)
    by = l3 * np.sin(theta3)
    theta2 = np.arctan2(by - ay, bx - ax)
    return theta2, theta3


def coupler_point(theta1, l2, l3):
    """Compute coupler point for given crank angle."""
    sol = fourbar_fk(theta1, L0, L1, l2, l3)
    if sol is None:
        return None
    theta2, _ = sol
    ax, ay = L1 * np.cos(theta1), L1 * np.sin(theta1)
    ux, uy = np.cos(theta2), np.sin(theta2)
    return np.array([ax + PX * ux - PY * uy,
                     ay + PX * uy + PY * ux])


def objective(x):
    """Path-synthesis objective."""
    l2, l3 = x
    path = []
    for theta1 in THETAS:
        pt = coupler_point(theta1, l2, l3)
        if pt is None:
            return 1e3
        path.append(pt)
    path = np.array(path)
    return CompareCurves(path[:, 0], path[:, 1],
                         TARGETS[:, 0], TARGETS[:, 1])


def grashof_g(x):
    """Grashof constraint: g(x) <= 0 means satisfied."""
    lengths = np.array([L0, L1, x[0], x[1]])
    s = np.sum(lengths)
    return (np.max(lengths) + np.min(lengths)) - (s - np.max(lengths) - np.min(lengths))

### Unconstrained BFGS (baseline)

We first run L-BFGS-B as in Tutorial 5, ignoring the Grashof constraint.

In [ ]:
x0 = np.array([1.0, 1.5])
bnds = [(0.3, 3.0), (0.3, 3.0)]

bfgs_path = [x0.copy()]
res_bfgs = minimize(objective, x0, method='L-BFGS-B', bounds=bnds,
                    callback=lambda xk: bfgs_path.append(xk.copy()),
                    options={'ftol': 1e-12, 'eps': 1e-7})
bfgs_path = np.array(bfgs_path)

print(f'L-BFGS-B (unconstrained):')
print(f'  x* = [{res_bfgs.x[0]:.4f}, {res_bfgs.x[1]:.4f}]')
print(f'  f  = {res_bfgs.fun:.4f}')
print(f'  g  = {grashof_g(res_bfgs.x):.4f}  '
      f'{"(violated!)" if grashof_g(res_bfgs.x) > 1e-6 else "(feasible)"}')

### SLSQP with Grashof constraint

Now let's use `method='SLSQP'` — scipy's SQP implementation — with the Grashof inequality constraint.

In [ ]:
sqp_path = [x0.copy()]

res_sqp = minimize(
    objective, x0, method='SLSQP', bounds=bnds,
    callback=lambda xk: sqp_path.append(xk.copy()),
    constraints={'type': 'ineq', 'fun': lambda x: -grashof_g(x)}
)
sqp_path = np.array(sqp_path)

print(f'SLSQP (Grashof-constrained):')
print(f'  x* = [{res_sqp.x[0]:.4f}, {res_sqp.x[1]:.4f}]')
print(f'  f  = {res_sqp.fun:.4f}')
print(f'  g  = {grashof_g(res_sqp.x):.4f}  (active constraint)')

### Trajectories with feasible region

Let's plot both optimization paths on the contour, with the infeasible region shaded.

In [ ]:
l2v = np.linspace(0.3, 2.5, 80)
l3v = np.linspace(0.3, 2.5, 80)
L2, L3 = np.meshgrid(l2v, l3v)
Z = np.vectorize(lambda a, b: objective([a, b]))(L2, L3)
G = np.vectorize(lambda a, b: grashof_g([a, b]))(L2, L3)

fig, ax = plt.subplots(figsize=(7, 5))
cf = ax.contourf(L2, L3, np.log1p(Z), levels=30, cmap='viridis')
# Shade infeasible region
ax.contourf(L2, L3, G, levels=[0, 10], colors=['red'], alpha=0.2)
ax.contour(L2, L3, G, levels=[0], colors=['red'],
           linewidths=2, linestyles='--')
# Trajectories
ax.plot(bfgs_path[:, 0], bfgs_path[:, 1], 'w.-', ms=6, lw=2,
        label='L-BFGS-B (unconstrained)')
ax.plot(sqp_path[:, 0], sqp_path[:, 1], 'r.-', ms=6, lw=2,
        label='SLSQP (constrained)')
ax.plot(*x0, 'ko', ms=8, label='start')
ax.set_xlabel(r'$\ell_2$')
ax.set_ylabel(r'$\ell_3$')
ax.set_title('BFGS vs. SQP: shaded region is Grashof-infeasible')
ax.legend(loc='upper left', fontsize=8)
plt.colorbar(cf, label=r'$\ln(1 + f)$')
plt.tight_layout()

### Why SQP finds a different optimum

The unconstrained BFGS path heads toward the global minimum of $f$ without regard for constraints. SQP, on the other hand, steers toward the best point on the Grashof boundary — the **constrained optimum**. The constraint forces a trade-off: a slightly higher objective value in exchange for a mechanism that can actually complete a full rotation.

Notice that SQP stays feasible (or returns to feasibility) throughout the iterations — the linearized constraints in each quadratic subproblem enforce this.

## Mechanism Hook

Let's compare the BFGS and SQP mechanisms. The SQP solution satisfies Grashof, guaranteeing full crank rotation.

In [ ]:
from dms.mechanisms.fourbar import FourBar

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
for ax, xopt, title in [
    (axes[0], res_bfgs.x, 'BFGS (unconstrained)'),
    (axes[1], res_sqp.x, 'SQP (Grashof)'),
]:
    l2, l3 = xopt
    mech = FourBar(L0, L1, l2, l3)
    path = np.array([coupler_point(t, l2, l3) for t in THETAS])
    mech.plot(np.deg2rad(90), ax=ax)
    ax.plot(TARGETS[:, 0], TARGETS[:, 1], 'r*', ms=10, label='targets')
    if path[0] is not None:
        ax.plot(path[:, 0], path[:, 1], 'b.-', label='coupler path')
    ax.set_aspect('equal')
    ax.legend(fontsize=8)
    ax.set_title(f'{title}\nf={objective(xopt):.3f}, '
                 f'g={grashof_g(xopt):.3f}')
plt.tight_layout()

SQP is powerful but requires smooth, differentiable constraints. In Tutorial 8, we explore **penalty methods** — a simpler alternative that converts constraints into objective-function penalties, trading algorithmic elegance for broader applicability.

---